# Create CBIR demo

In this notebook, we create a public access bucket, which will store a:
- connectome_data.csv file - holds the query patient and matching patient embedding, as well as UIDs and OHIF urls
- index.html - holds code on how to create the website for exploring the foundation model embeddings

The connectome_data.csv will be created from the connectome_data BigQuery table. See other notebook for details about how to create this.

The index.html file is currently uploaded from Google Drive to the public access bucket, later this can be done in GitHub release attachments.

Deepa Krishnaswamy

Brigham and Women's Hospital

March 2026

# Parameterization

In [1]:
#@title Enter your Project ID here
# initialize this variable with your Google Cloud Project ID!
project_name = "idc-external-018" #@param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = project_name

!gcloud config set project $project_name

from google.colab import auth
auth.authenticate_user()

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



# Setup

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
PROJECT_ID = project_name
DATASET_ID = "nlst_cbir_connectome" # For BigQuery
BUCKET_ID = "nlst_cbir_connectome" # Storage

connectome_bq_table = '.'.join([PROJECT_ID, DATASET_ID, "connectome_data"])
print("connectome BQ table: " + str(connectome_bq_table))

connectome BQ table: idc-external-018.nlst_cbir_connectome.connectome_data


In [11]:
from google.cloud import bigquery
import os
import pandas as pd
import numpy as np
import json

In [ ]:
client = bigquery.Client(project=PROJECT_ID)

# Create csv file for connectome data

In [5]:
# Create a csv file from the BQ table

query = f"""
          SELECT
            *
          FROM
            `{connectome_bq_table}`
        """
result = client.query(query)
connectome_data_df = result.to_dataframe(create_bqstorage_client=True)

In [6]:
# Save this csv file

connectome_data_df.to_csv("/content/connectome_data.csv")

# Create a public access bucket and upload the connectome_data.csv file

In [7]:
# Create a public access bucket

!gsutil mb gs://$BUCKET_ID

# Create a policy so that everyone can view

!gcloud storage buckets add-iam-policy-binding gs://$BUCKET_ID \
    --member=allUsers \
    --role=roles/storage.objectViewer

Creating gs://nlst_cbir_connectome/...
bindings:
- members:
  - projectEditor:idc-external-018
  - projectOwner:idc-external-018
  role: roles/storage.legacyBucketOwner
- members:
  - projectViewer:idc-external-018
  role: roles/storage.legacyBucketReader
- members:
  - allUsers
  role: roles/storage.objectViewer
etag: CAI=
kind: storage#policy
resourceId: projects/_/buckets/nlst_cbir_connectome
version: 1


In [9]:
# Copy this csv file to the bucket

!gsutil cp "/content/connectome_data.csv" gs://$BUCKET_ID

Copying file:///content/connectome_data.csv [Content-Type=text/csv]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

-
Operation completed over 1 objects/246.7 MiB.                                    


# Set permissions on files in the bucket

In [12]:
# Create cors.json file to set permissions for the connectome csv file with precomputed closest matches

# Your data to be stored in the JSON file
data = [{"origin": ["*"],"method": ["GET"],"responseHeader": ["Content-Type"],"maxAgeSeconds": 3600}]

# Specify the filename
filename = "/content/cors.json"

# Write the data to the JSON file
with open(filename, "w") as f:
    json.dump(data, f, indent=4) # Using indent=4 makes the file human-readable
print(f"JSON file '{filename}' created.")

JSON file '/content/cors.json' created.
Setting CORS on gs://nlst_cbir_connectome/...


In [19]:
# Copy the index.html file from Google Drive to the bucket
# Later this can change

!gsutil cp "/content/gdrive/MyDrive/Colab Notebooks/nlst-cbir-connectome/site/index.html" gs://$BUCKET_ID/site/index.html

Copying file:///content/gdrive/MyDrive/Colab Notebooks/nlst-cbir-connectome/site/index.html [Content-Type=text/html]...
- [1 files][ 11.7 KiB/ 11.7 KiB]                                                
Operation completed over 1 objects/11.7 KiB.                                     


In [20]:
# Set the cors permissions

# !gsutil cors set /content/cors.json gs://$BUCKET_ID/connectome_data.csv
# !gsutil cors set /content/cors.json gs://$BUCKET_ID/site/index.html

!gsutil -m acl ch -u AllUsers:R gs://$BUCKET_ID/connectome_data.csv

!gsutil -m acl ch -u AllUsers:R gs://$BUCKET_ID/site/index.html

!gsutil iam ch allUsers:objectViewer gs://$BUCKET_ID

!gsutil cors set /content/cors.json gs://$BUCKET_ID

No changes to gs://nlst_cbir_connectome/connectome_data.csv
Updated ACL on gs://nlst_cbir_connectome/site/index.html
No changes made to gs://nlst_cbir_connectome/
Setting CORS on gs://nlst_cbir_connectome/...
